# 10 · Differential geometry on the sphere

`omnibias-geometry` builds the Riemannian toolkit on top of `omnibias-fields`:
the metric, Christoffel symbols, the **Laplace–Beltrami operator**, and
Riemann / Ricci / scalar curvature. Field derivatives are exact closed forms;
the metric derivatives come from autodiff of the analytic metric.

We use the round 2-sphere of radius `R` in `(θ, φ)` coordinates, with metric
`g = diag(R², R² sin²θ)`. Two facts we will recover numerically:

- scalar curvature `R_scalar = 2/R²` (constant over the sphere),
- `Δ_g cos θ = −(2/R²) cos θ` (`cos θ` is a degree-1 spherical-harmonic
  eigenfunction of the Laplace–Beltrami operator).

In [ ]:
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, GOOD
from _fields import make_field, Cos, Const
set_style()

torch.set_default_dtype(torch.float64)

from omnibias.geometry import ManifoldSpec, MetricSpec
from omnibias.geometry.torch import ops as geo
from omnibias.fields.torch import _ops_dispatch as dispatch

R = 1.5


def sphere_metric(x):  # x: (2,) = (theta, phi)
    theta = x[0]
    z = 0.0 * theta
    return torch.stack([
        torch.stack([R**2 + z, z]),
        torch.stack([z, R**2 * torch.sin(theta) ** 2]),
    ])


sphere = ManifoldSpec("S2", 2, MetricSpec(sphere_metric, dim=2, name="round_sphere"))
print(sphere)

## Curvature

`scalar_curvature` should be the constant `2/R²` everywhere, and the Ricci
tensor should equal `g/R²` (an Einstein manifold).

In [ ]:
theta = torch.linspace(0.4, np.pi - 0.4, 25)
phi = torch.full_like(theta, 0.7)
coords = torch.stack([theta, phi], dim=-1)

scal = geo.scalar_curvature(coords, sphere)
print(f"scalar curvature: min={scal.min():.10f}  max={scal.max():.10f}")
print(f"expected 2/R^2  = {2.0 / R**2:.10f}")

ric = geo.ricci_tensor(coords, sphere)
g = geo.metric(coords, sphere)
print("max |Ricci - g/R^2| =", float((ric - g / R**2).abs().max()))

## Laplace–Beltrami eigenfunction

`cos θ` is a degree-1 spherical harmonic, so `Δ_g cos θ = −(2/R²) cos θ`. We
build the *exact* field `f(θ, φ) = cos θ` and apply `laplace_beltrami`.

In [ ]:
field = make_field(
    ("theta", "phi"),
    {"f": (Cos(1.0, 1.0, xp=torch), Const(1.0))},  # f = cos(theta) * 1
    dispatch,
)
state = field(coords)
lb = geo.laplace_beltrami(state, "f", sphere)
expected = -2.0 / R**2 * torch.cos(theta)
print("max |Δ_g f - (-2/R^2) cosθ| =", float((lb - expected).abs().max()))

In [ ]:
fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.2))

axl.plot(theta.numpy(), scal.numpy(), color=PRIMARY, label=r"$R_{\mathrm{scalar}}$")
axl.axhline(2.0 / R**2, ls="--", color=ACCENT, label=r"$2/R^2$")
axl.set_xlabel(r"$\theta$")
axl.set_title("Scalar curvature is constant")
axl.legend()

axr.plot(theta.numpy(), lb.detach().numpy(), color=GOOD, label=r"$\Delta_g f$")
axr.plot(theta.numpy(), expected.numpy(), "--", color=ACCENT,
         label=r"$-\frac{2}{R^2}\cos\theta$")
axr.set_xlabel(r"$\theta$")
axr.set_title("Laplace–Beltrami eigenfunction")
axr.legend()

plt.tight_layout()
plt.show()

## Takeaway

The metric → connection → curvature pipeline and the Laplace–Beltrami operator
match analytic geometry to machine precision, with torch/jax parity. Swap in
your own metric callable (torus, Lorentzian, a learned chart) and the same ops
apply — the foundation for PINNs and VMC on curved domains.

Next: **[11 · Fractional diffusion](11_fractional_diffusion.ipynb)**.